# Step 5: Train the improved model
Same data split, seed, image size, optimiser and augmentation as the baseline (`configs/train.yaml`). Only the architecture differs.

In [ ]:
# Set True for a fast CPU sanity run (3 epochs, 320 px, 25% of train). False = full experiment.
QUICK_TEST = False
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # works from VS Code either way
sys.path.insert(0, str(ROOT / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config(quick=QUICK_TEST)
RUNS = md.ROOT / "runs"
YOLO_DIR = md.ROOT / cfg["dataset"]["yolo_dir"]
DATA_YAML = YOLO_DIR / "data.yaml"
import yaml, torch
assert DATA_YAML.exists(), "Run 02_dataset_split.ipynb first"
names = list(yaml.safe_load(open(DATA_YAML))["names"].values())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU) - use QUICK_TEST = True")

In [ ]:
BEST_PROP = train.train_improved(cfg, DATA_YAML, RUNS, name="proposed_yolov8m", nc=len(names))
print("Saved:", BEST_PROP)

In [ ]:
import matplotlib.pyplot as plt
cb, cp = evaluate.load_curve(RUNS, "baseline_yolov8m"), evaluate.load_curve(RUNS, "proposed_yolov8m")
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, (col, t) in zip(ax, [("train/box_loss", "Train box loss"), ("metrics/mAP50(B)", "Val mAP@50"), ("metrics/mAP50-95(B)", "Val mAP@50-95")]):
    a.plot(cb["epoch"], cb[col], label="Baseline"); a.plot(cp["epoch"], cp[col], label="Improved")
    a.set_title(t); a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.show()

### Do the learned gates move away from zero?
If every `gamma` stays near 0, the attention blocks are effectively unused.

In [ ]:
import sys
from ultralytics import YOLO
from mri_detection import modules as cm
m = YOLO(str(RUNS / "proposed_yolov8m" / "weights" / "best.pt")).model
print("CBAM gamma:", [round(float(b.gamma), 4) for b in m.modules() if isinstance(b, cm.CBAMAttention)])
print("Fusion weights:", [[round(float(w), 3) for w in b.w] for b in m.modules() if isinstance(b, cm.WeightedConcat)])

Next: `06_evaluation.ipynb`.